In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rtnhiuthanhthao/dch-anh-sang-vit/kaggle/working/envi-model/config.json
/kaggle/input/datasets/rtnhiuthanhthao/dch-anh-sang-vit/kaggle/working/envi-model/vocab.json
/kaggle/input/datasets/rtnhiuthanhthao/dch-anh-sang-vit/kaggle/working/envi-model/tokenizer_config.json
/kaggle/input/datasets/rtnhiuthanhthao/dch-anh-sang-vit/kaggle/working/envi-model/source.spm
/kaggle/input/datasets/rtnhiuthanhthao/dch-anh-sang-vit/kaggle/working/envi-model/model.safetensors
/kaggle/input/datasets/rtnhiuthanhthao/dch-anh-sang-vit/kaggle/working/envi-model/target.spm
/kaggle/input/datasets/rtnhiuthanhthao/dch-anh-sang-vit/kaggle/working/envi-model/generation_config.json


In [13]:
import os
print(os.listdir('/kaggle/input'))

['datasets']


In [14]:
print(os.listdir('/kaggle/input/datasets'))

['rtnhiuthanhthao']


In [15]:
print(os.listdir('/kaggle/input/datasets/rtnhiuthanhthao'))

['dch-anh-sang-vit']


In [18]:
import os
print(os.listdir('/kaggle/input/datasets/rtnhiuthanhthao/dch-anh-sang-vit')) 

['kaggle']


In [23]:
import os
print(os.listdir('/kaggle/input/datasets/rtnhiuthanhthao/dch-anh-sang-vit/kaggle/working')) 

['envi-model']


In [24]:
model_path = '/kaggle/input/datasets/rtnhiuthanhthao/dch-anh-sang-vit/kaggle/working/envi-model' 

In [25]:
print(os.listdir(model_path))

['config.json', 'vocab.json', 'tokenizer_config.json', 'source.spm', 'model.safetensors', 'target.spm', 'generation_config.json']


In [26]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path) 

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [27]:
!pip install -q transformers datasets sentencepiece sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.6 MB/s eta 0:00:00


In [28]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
) 

In [29]:
dataset = load_dataset("thainq107/iwslt2015-en-vi")

print(dataset) 

README.md:   0%|          | 0.00/522 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/181k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/133317 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1268 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1268 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['en', 'vi'],
        num_rows: 133317
    })
    validation: Dataset({
        features: ['en', 'vi'],
        num_rows: 1268
    })
    test: Dataset({
        features: ['en', 'vi'],
        num_rows: 1268
    })
})


In [30]:
train_dataset = dataset["train"].select(range(35000))
valid_dataset = dataset["validation"].select(range(1200)) 

print(len(train_dataset))
print(len(valid_dataset)) 

35000
1200


In [31]:
model_name = "Helsinki-NLP/opus-mt-en-vi"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)   

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/289M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

In [32]:
prefix = "translate English to Vietnamese: "

def preprocess(example):

    inputs = [prefix + x for x in example["en"]]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        text_target=example["vi"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs 

In [33]:
tokenized_train = train_dataset.map(
    preprocess,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_valid = valid_dataset.map(
    preprocess,
    batched=True,
    remove_columns=valid_dataset.column_names
) 

Map:   0%|          | 0/35000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

In [34]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [35]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./model",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    num_train_epochs=5,

    predict_with_generate=True,

    logging_steps=200,

    save_total_limit=2,

    fp16=True
) 

In [36]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,

    data_collator=data_collator
) 

In [37]:
trainer.train()

Step,Training Loss
200,1.711699
400,1.577955
600,1.565866
800,1.556324
1000,1.506710
1200,1.503875
1400,1.500761
1600,1.539722
1800,1.514340
2000,1.490433


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=21875, training_loss=1.2531793289620536, metrics={'train_runtime': 1639.2961, 'train_samples_per_second': 106.753, 'train_steps_per_second': 13.344, 'total_flos': 2806790971981824.0, 'train_loss': 1.2531793289620536, 'epoch': 5.0})

In [38]:
sentences = [
    "I love machine learning.",
    "This is a beautiful day.",
    "We will meet tomorrow.",
    "Artificial intelligence is changing the world."
]

for s in sentences:
    input_text = "translate English to Vietnamese: " + s
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_length=64, num_beams=5)

    print("EN:", s)
    print("VI:", tokenizer.decode(outputs[0], skip_special_tokens=True))
    print() 

EN: I love machine learning.
VI: Tôi yêu thích việc học hỏi bằng máy .

EN: This is a beautiful day.
VI: Đây là một ngày đẹp trời .

EN: We will meet tomorrow.
VI: Chúng ta sẽ gặp nhau vào ngày mai .

EN: Artificial intelligence is changing the world.
VI: Trí thông minh nhân tính đang thay đổi thế giới .



In [39]:
text = "Sometimes I feel like my only friend."

input_text = "translate English to Vietnamese: " + text

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_length=64,
    num_beams=5
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True)) 

Đôi khi tôi cảm thấy mình là bạn duy nhất của mình .


In [40]:
sentences = [
    "Is this the real life? Is this just fantasy?",
    "Caught in a landslide, no escape from reality.",
    "Open your eyes, look up to the skies and see",
    "AI'm just a poor boy, I need no sympathy.",
    "Because I'm easy come, easy go",
    "Little high, little low",
    "Any way the wind blows doesn't really matter to me, to me"
]

for s in sentences:
    input_text = "translate English to Vietnamese: " + s
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_length=64, num_beams=5)

    print("EN:", s)
    print("VI:", tokenizer.decode(outputs[0], skip_special_tokens=True))
    print() 

EN: Is this the real life? Is this just fantasy?
VI: Đây có phải là cuộc sống thực sự ? Đây có phải là sự tưởng tượng ?

EN: Caught in a landslide, no escape from reality.
VI: Bị rơi trong một vụ lở đất , không thoát khỏi thực tế .

EN: Open your eyes, look up to the skies and see
VI: Hãy mở to mắt , nhìn lên bầu trời và nhìn thấy

EN: AI'm just a poor boy, I need no sympathy.
VI: AI chỉ là một cậu bé tội nghiệp , tôi không cần sự cảm thông .

EN: Because I'm easy come, easy go
VI: Bởi vì tôi dễ dàng đến , dễ dàng đi .

EN: Little high, little low
VI: Cao nhẹ , thấp

EN: Any way the wind blows doesn't really matter to me, to me
VI: Bất kỳ cách nào mà gió thổi cũng không ảnh hưởng đến tôi , đối với tôi .



In [45]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [46]:
import evaluate

bleu = evaluate.load("bleu")

In [47]:
predictions = []
references = []

num_samples = min(200, len(valid_dataset))

for example in valid_dataset.select(range(num_samples)):

    input_text = example["en"]

    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(inputs["input_ids"], max_length=128)

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    ref = example["vi"]

    predictions.append(pred)
    references.append([ref])

result = bleu.compute(predictions=predictions, references=references)

print("BLEU:", result["bleu"] * 100) 

BLEU: 30.819570529098606


In [48]:
model.save_pretrained("/kaggle/working/envi-model")
tokenizer.save_pretrained("/kaggle/working/envi-model")  

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/envi-model/tokenizer_config.json',
 '/kaggle/working/envi-model/vocab.json',
 '/kaggle/working/envi-model/source.spm',
 '/kaggle/working/envi-model/target.spm',
 '/kaggle/working/envi-model/added_tokens.json')

In [49]:
!zip -r envi-model.zip /kaggle/working/envi-model

  adding: kaggle/working/envi-model/ (stored 0%)
  adding: kaggle/working/envi-model/config.json (deflated 63%)
  adding: kaggle/working/envi-model/source.spm (deflated 51%)
  adding: kaggle/working/envi-model/vocab.json (deflated 70%)
  adding: kaggle/working/envi-model/target.spm (deflated 50%)
  adding: kaggle/working/envi-model/model.safetensors (deflated 7%)
  adding: kaggle/working/envi-model/generation_config.json (deflated 43%)
  adding: kaggle/working/envi-model/tokenizer_config.json (deflated 68%)
